# Training: 30-Class EfficientNet-B0

Fine-tune an EfficientNet-B0 (from scratch) for 30-class fruit classification (FIDS30).

In [ ]:
import timm
import torch
import torch.nn as nn

NUM_CLASSES = 30
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES) # change pretrained to False to train from scratch

print(f"Model: {model.default_cfg['architecture']}")
print(f"Classifier head: {model.get_classifier()}")

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Model: efficientnet_b0
Classifier head: Linear(in_features=1280, out_features=30, bias=True)


In [3]:
from torchvision import datasets
from torch.utils.data import DataLoader

config = timm.data.resolve_model_data_config(model)
train_transform = timm.data.create_transform(**config, is_training=True)
val_transform = timm.data.create_transform(**config, is_training=False)

train_ds = datasets.ImageFolder("PrepData/Training", transform=train_transform)
val_ds = datasets.ImageFolder("PrepData/Validation", transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

print(f"Classes: {train_ds.classes}")
print(f"Training samples: {len(train_ds)}, Validation samples: {len(val_ds)}")

Classes: ['acerolas', 'apples', 'apricots', 'avocados', 'bananas', 'blackberries', 'blueberries', 'cantaloupes', 'cherries', 'coconuts', 'figs', 'grapefruits', 'grapes', 'guava', 'kiwifruit', 'lemons', 'limes', 'mangos', 'olives', 'oranges', 'passionfruit', 'peaches', 'pears', 'pineapples', 'plums', 'pomegranates', 'raspberries', 'strawberries', 'tomatoes', 'watermelons']
Training samples: 582, Validation samples: 194


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
print(f"Device: {device}") # cuda means i am training on gpu

Device: cuda


In [5]:
from tqdm.auto import tqdm

epochs = 10
for epoch in range(epochs):
    # Training Phase
    model.train()
    train_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation Phase
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / len(val_ds)
    print(f"Epoch {epoch+1}: Loss = {train_loss/len(train_loader):.4f}, Val Acc = {accuracy:.2f}%")

Epoch 1/10:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1: Loss = 4.1692, Val Acc = 14.95%


Epoch 2/10:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 2: Loss = 2.7723, Val Acc = 37.11%


Epoch 3/10:   0%|          | 0/10 [00:00<?, ?it/s]

...

Epoch 8: Loss = 0.7213, Val Acc = 76.80%


Epoch 9/10:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 9: Loss = 0.7444, Val Acc = 79.90%


Epoch 10/10:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 10: Loss = 0.5930, Val Acc = 81.44%


When setting ```pretrained=False```, i.e. training the model from scratch, the accuracy will be around 7 %, it only slightly increases over the epochs. 

In [6]:
torch.save(model.state_dict(), "fids30_classifier_30cls_b0.pth")